# HALTools


GisMap exposes two helpers to compare what different databases know about
a researcher and to spot duplicates inside a single database:

- `diff_sources(a, b)` lists publications present in `a` but not in `b`,
  and vice-versa. Useful when you want to manually update one database
  using inputs from another one.
- `find_duplicates(a)` groups publications that look like the same paper
  inside `a` (e.g. an entry created by you and a near-duplicate created
  by a co-author).

A frequent use is to audit the database you can actually edit, and
compare it with a more remote one. For researchers based in France, the
editable database is usually [HAL](https://hal.science/) — hence the
nickname *HALTools*. The methods themselves are not HAL-specific.

Typical use cases:

- Spot duplicates of the same publication that you and a co-author
  registered independently.
- Verify that all your DBLP-indexed publications have a counterpart in
  HAL (mandatory for most French academic labs, easy to forget).
- For researchers with a partial HAL footprint (e.g. several `pid`s and
  no unified `HAL-ID`), HALTools helps reconcile what is found by which
  identifier.

The rest of this notebook walks through illustrated examples.


:::note
You do not need a full `LabMap` to use HALTools — a `LabAuthor` is enough
(technically `SourcedAuthor` would do, but `LabAuthor` is more convenient).
:::


:::important
HALTools surface entries to **check**, not entries to **change**. As we
will see below, many flagged items are perfectly fine and need no action.
:::


## Céline — a clean researcher

We start with [Céline Comte](https://comte-celine.fr/), whose HAL and
DBLP profiles are well-maintained. The differences we find should be
explainable, not actionable.


In [ ]:
from gismap.lab.lab_author import LabAuthor

celine = LabAuthor("Céline Comte")
celine.auto_sources()

### Comparing HAL and LDB


In [ ]:
print(celine.diff_sources("hal", "ldb"))

Reading the output:

- Most HAL-only papers are written in French, typically Algotel
  conference papers aimed at the French-speaking community. DBLP rarely
  indexes French-only papers (with a few exceptions like
  [JIAF 2023](https://dblp.org/db/conf/jiaf/jiaf2023.html#conf/jiaf/CaizerguesDM23)).
  Nothing wrong here.
- *Networks of multi-server queues with parallel processing* (LDB-only):
  likely an early research report later absorbed into a journal paper
  with a different title.
- *Online Stochastic Matching: A Polytope Perspective* (HAL) and
  *Stochastic dynamic matching: A mixed graph-theory and linear-algebra
  approach* (LDB) are actually the same article. HAL stores the latest
  version (newer date and title), DBLP/LDB keeps the original. Merging
  these would be a hard feature for a small payoff — don't expect it
  soon.
- *Score-Aware Policy-Gradient Methods…* (LDB-only): same pattern, a
  report later published in a journal under a slightly different title.
  Could be merged with a more lenient title-similarity threshold, but
  that knob is not exposed in the public API yet.

Bottom line: everything is fine.


### Duplicates inside HAL


In [ ]:
print(celine.find_duplicates("hal"))

Both groups are report-to-conference or conference-to-journal
lifecycles — no actual issue. GisMap merges these duplicates
automatically when building a map, so no action is needed.


## François — a messier real-world case

[François Durand](https://francoisdurand.fr/) has a richer history,
with French papers, posters, and a few real gaps. We pin the
HAL `pid` explicitly to avoid homonyms.


In [ ]:
francois = LabAuthor("François Durand (hal:fradurand)")
francois.auto_sources()

### Spotting actual gaps in HAL

In [ ]:
print(francois.diff_sources("hal", "ldb"))

**Only in HAL** — all explainable:

- French papers (DBLP rarely indexes them), including the PhD thesis (written in French).
- Items without proper proceedings (posters, *Voter Autrement* dataset
  reports).
- *Coalitional manipulation of voting rules…* — published in an
  economics venue outside DBLP's scope.

**Only in LDB** — mixed bag:

- The two beamforming papers and the Condorcet paper *should* be in HAL
  but were forgotten. These are real action items.
- *L'abus de comparaisons est mauvais pour la santé*: the HAL entry is
  the **whole proceedings** of the conference, with the PC chairs as
  authors. DBLP somehow extracted the individual papers from HAL even
  though HAL itself does not split them.
- *Detection of Horse Locomotion…*: a rare DBLP confusion. The `pid`
  `38/11269` is supposed to point to a single person, but this
  particular paper was authored by a homonym. The only fix is to email
  DBLP directly (`dblp@dagstuhl.de`); response time can be slow.
- *Sorting wild pigs* / *Trier des cochons sauvages*: a French paper
  that was OK to file in French because it carries an English title and
  abstract. Nothing to patch.


### Duplicates: same paper, different lifecycle stages


In [ ]:
print(francois.find_duplicates("hal"))

All five groups are benign:

- **Group 1** — three distinct documents around the same *Voter
  Autrement 2017* experiment.
- **Group 2** — a poster summarizing a longer report.
- **Group 3** — two evolutions of the same paper.
- **Group 4** — short report vs. short announcement of the same tool.
- **Group 5** — report-to-conference lifecycle.


## Fabien — many duplicates, one real issue

On older HAL profiles, the report → conference → journal lifecycle has
had time to accumulate. The interesting question is whether a real
duplicate hides among the noise.


In [ ]:
fabien = LabAuthor("Fabien Mathieu")
fabien.auto_sources()

Focus on duplicates:


In [ ]:
print(fabien.find_duplicates("hal"))

Groups 1 to 13 follow the usual lifecycle pattern (report → conference
→ journal/chapter) and need no action.

**Group 14 is the real issue**: *Deciding and verifying network
properties locally with few output bits* exists twice in HAL because two
co-authors entered it independently. The fix is to ask HAL support to
merge the two entries.


## Élie — when the HAL `pid` is too strict

Élie de Panafieu has a low HAL footprint and a more idiosyncratic
profile. Default settings will only catch part of his work — HALTools
is exactly the tool to diagnose what is missing and why.


By default, GisMap picks the most specific HAL identifier it can find:


In [ ]:
elie = LabAuthor("Élie de Panafieu")
elie.auto_sources("hal")
elie.sources

In [ ]:
print("\n".join(str(p) for p in elie.get_publications().values()))


Only one publication. The HAL `pid` is unambiguous but too narrow:
many of Élie's HAL entries are not attached to that `pid`. We can
confirm this by comparing `pid`-based and fullname-based searches:


In [ ]:
elie = LabAuthor("Élie de Panafieu (hal:1319887, hal:fullname)")
print(elie.diff_sources(0, 1))

The fullname search picks up five more HAL entries that the `pid` misses.
The fix is to switch to fullname (the `hal:fullname` flag is documented
in the [FAQ](../faq.ipynb)):


In [ ]:
elie = LabAuthor("Élie de Panafieu (hal:fullname)")
elie.auto_sources()

Comparing HAL (now in fullname mode) against LDB:


In [ ]:
print(elie.diff_sources("hal", "ldb"))

Élie's exposure is mostly on DBLP/LDB — and that is fine. HAL deposit
is only mandatory for researchers affiliated with a French academic
institution; the LDB-only entries here are not action items.


## When to act

Across the four examples, only a handful of items called for action:

- **Missing HAL deposits** (François: beamforming and Condorcet papers).
- **Genuine HAL duplicates** entered by two co-authors (Fabien: *Deciding
  and verifying network properties…*).
- **Wrong DBLP attribution to a homonym** (François: *Horse Locomotion*).
- **Identifier strategy** (Élie: switch from `pid` to fullname).

Everything else — lifecycle duplicates, French-only papers, reports
absorbed into journals — is normal database life that GisMap already
handles when building a map.
